# 07. Hyperparameter Tuning with GridSearchCV

Use the Qatar used-car dataset from the regularization example to choose a Ridge or Lasso model and its regularization strength (`alpha`).

1. Hold out 25% of the data for a final test.
2. Compare candidate models and `alpha` values using cross-validation on the training set.
3. Refit the best pipeline on all training data and evaluate it once on the test set.

The dataset is small, so scores and the selected model can change with the train/test split or cross-validation folds.

In [16]:
import numpy as np
import pandas as pd
from sklearn.linear_model import Lasso, LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV, KFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

feature_cols = ['age_years', 'mileage_km', 'engine_size_l']
df = pd.read_csv('data/qatar_used_cars.csv')
X = df[feature_cols]
y = df['price_qar']

# Keep the test set separate from model selection.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)
print(f'Training samples: {len(X_train)} | Test samples: {len(X_test)}')
display(df.head())

Training samples: 18 | Test samples: 6


,model,age_years,mileage_km,engine_size_l,price_qar
0,Toyota Camry,1,18000,2.5,92000
1,Toyota Camry,2,42000,2.5,81000
2,Toyota Camry,3,35000,2.5,76000
3,Toyota Camry,4,78000,2.5,63000
4,Toyota Camry,6,110000,2.5,48000


## Cross-validated grid search

`GridSearchCV` trains each candidate on four folds and validates it on the fifth, repeating this across five folds. It selects the candidate with the lowest average validation root mean squared error (RMSE). Scikit-learn reports negative RMSE so that larger scores are always better.

The `Pipeline` fits `StandardScaler` separately within each training fold, preventing information from the validation fold from leaking into preprocessing. Ridge and Lasso have different objectives, so their `alpha` values are searched on separate scales.

In [17]:
# Scale features inside each fold, then fit a regression model.
# Ridge is a placeholder: the grid below also replaces it with Lasso.
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', Ridge()),
])

# Try each model with its own values of alpha (regularization strength).
# model__alpha refers to the alpha parameter of the pipeline's model step.
param_grid = [
    {
        'model': [Ridge()],
        'model__alpha': [0.01, 0.1, 1.0, 10.0, 100.0],
    },
    {
        'model': [Lasso(max_iter=100000)],
        'model__alpha': [1.0, 10.0, 100.0, 1000.0, 10000.0],
    },
]

# cv can be simply an integer, but we want to shuffle the data for each fold.
cv = KFold(n_splits=5, shuffle=True, random_state=42)
search = GridSearchCV(
    pipeline,
    param_grid,
    scoring='neg_root_mean_squared_error',
    cv=cv,
    refit=True,
    return_train_score=True,
    error_score='raise',
)
search.fit(X_train, y_train)

print(f'Best model: {type(search.best_estimator_.named_steps["model"]).__name__}')
print(f'Best alpha: {search.best_params_["model__alpha"]:g}')
# Scikit-learn stores RMSE as a negative score; negate it for display.
print(f'Best mean validation RMSE: {-search.best_score_:,.0f} QAR')

Best model: Ridge
Best alpha: 10
Best mean validation RMSE: 39,203 QAR


## Compare the candidates

The table ranks candidates by mean validation RMSE; lower is better. The standard deviation describes variation across folds, and the training RMSE helps reveal a gap between training and validation performance. These are averages of fold RMSEs, not RMSEs computed from pooled predictions.

In [21]:
# Each candidate's parameters include its model object.
model_names = []
for params in search.cv_results_['params']:
    model = params['model']
    model_name = type(model).__name__  # For example, "Ridge" or "Lasso".
    model_names.append(model_name)

# Turn the search scores into a table, ordered from best to worst.
# Negate the mean scores to show RMSE as a positive value.
cv_results = pd.DataFrame({
    'Model': model_names,
    'Alpha': search.cv_results_['param_model__alpha'],
    'Mean train RMSE (QAR)': -search.cv_results_['mean_train_score'],
    'Mean validation RMSE (QAR)': -search.cv_results_['mean_test_score'],
    'Validation RMSE std (QAR)': search.cv_results_['std_test_score'],
    'Rank': search.cv_results_['rank_test_score'],
}).sort_values('Rank').reset_index(drop=True)

display(cv_results.round(2))

,Model,Alpha,Mean train RMSE (QAR),Mean validation RMSE (QAR),Validation RMSE std (QAR),Rank
0,Ridge,10.00,34391.68,39203.27,15849.85,1
1,Ridge,1.00,26336.97,39686.73,10430.07,2
2,Lasso,10000.00,29721.11,40715.02,15937.22,3
3,Ridge,0.10,26064.25,40748.12,10858.02,4
4,Lasso,1000.00,26092.31,40858.19,11232.49,5
5,Lasso,100.00,26051.68,41050.27,11032.08,6
6,Ridge,0.01,26051.28,41058.86,11002.55,7
7,Lasso,10.00,26051.11,41084.40,11021.99,8
8,Lasso,1.00,26050.97,41111.07,11027.65,9
9,Ridge,100.00,57506.21,52769.90,28435.25,10


## Final evaluation on the held-out test set

With `refit=True`, the selected pipeline is already refitted on all training data. Compare it with the unregularized linear regression baseline from the previous notebook using MAE, RMSE, and $R^2$. The test set is for final reporting, not for choosing another model or grid. Coefficients are in QAR per one standard deviation of each feature.

In [19]:
# Fit an unregularized baseline for comparison.
baseline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LinearRegression()),
])
baseline.fit(X_train, y_train)

# The best pipeline was already refitted by GridSearchCV.
best_pipeline = search.best_estimator_
results = []
for name, estimator in [('Linear regression baseline', baseline),
                        ('Tuned model', best_pipeline)]:
    predictions = estimator.predict(X_test)
    results.append({
        'Model': name,
        'Test MAE (QAR)': mean_absolute_error(y_test, predictions),
        'Test RMSE (QAR)': np.sqrt(mean_squared_error(y_test, predictions)),
        'Test R²': r2_score(y_test, predictions),
    })

display(pd.DataFrame(results).round(2))

# Coefficients describe the effect of one standard deviation of each feature.
coefficients = pd.DataFrame({
    'Feature': feature_cols,
    'Coefficient (QAR per standard deviation)': best_pipeline.named_steps['model'].coef_,
})
display(coefficients.round(2))

,Model,Test MAE (QAR),Test RMSE (QAR),Test R²
0,Linear regression baseline,15773.17,17166.15,0.88
1,Tuned model,19144.88,21181.08,0.82


,Feature,Coefficient (QAR per standard deviation)
0,age_years,-15190.46
1,mileage_km,-14206.99
2,engine_size_l,30695.75
